In [22]:
import json
import os
import requests
from base64 import b64encode
from datetime import datetime, timezone
from getpass import getpass
from netCDF4 import Dataset
from urllib.parse import urlparse
from pathlib import Path

# URL for the CEDA Token API service
TOKEN_URL = "https://ceda.ac.uk"
# Location on the filesystem to store a cached download token
TOKEN_CACHE = os.path.expanduser(os.path.join("~", ".cedatoken"))


In [23]:
MY_CEDA_TOKEN = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICI4ZjhmaUpyaUtDY3hmaHhzdU5vazVEekdJdFZ4amhhTWNJa05ZX2U4MnhJIn0.eyJleHAiOjE3ODI0NzAzMjgsImlhdCI6MTc4MjIxMTEyOCwianRpIjoiNmU1ZjhkMDktNDI2NC00Zjg5LWE4NjMtMGI1MTkxZmZkNGMxIiwiaXNzIjoiaHR0cHM6Ly9hY2NvdW50cy5jZWRhLmFjLnVrL3JlYWxtcy9jZWRhIiwic3ViIjoiYzkzMTMxOWQtOTdkMC00NzZjLTkwM2EtOTBlYmU0NTMxZWRmIiwidHlwIjoiQmVhcmVyIiwiYXpwIjoic2VydmljZXMtcG9ydGFsLWNlZGEtYWMtdWsiLCJzZXNzaW9uX3N0YXRlIjoiM2QzM2I4MmItMzdhZS00ZGQyLWJkN2UtYmEzZTNkZjI1ZWUzIiwiYWNyIjoiMSIsInNjb3BlIjoiZW1haWwgb3BlbmlkIHByb2ZpbGUgZ3JvdXBfbWVtYmVyc2hpcCIsInNpZCI6IjNkMzNiODJiLTM3YWUtNGRkMi1iZDdlLWJhM2UzZGYyNWVlMyIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJuYW1lIjoiUmF1bCBDb3JyZWEiLCJwcmVmZXJyZWRfdXNlcm5hbWUiOiJyYWNvbzIwMyIsImdpdmVuX25hbWUiOiJSYXVsIiwiZmFtaWx5X25hbWUiOiJDb3JyZWEiLCJlbWFpbCI6InJhdWwuY29ycmVhb2NhbmFzQGVzc2V4Lmdvdi51ayJ9.n1sB-ZE8n1I_q3w_521Jsm85c52wUYOhb7bPw8fMeIoPoKrLhbqJDy_EwjDi_mtHe7Dm1G8zf-GPuK7kWhCFb5ndQsAqFtR3yGeRoFSgLg50d45frbeO2QONlNXY2hZs6vngnCN7e4PdgG-ymW-QdjFjXeVRRbFPKut1gfP2KkRBv-oJMFutytidiCIfII8Y-N20BpePFrXi35jxDqXQlDTKtJR4ckkMacsxGqsJvrI5Jyeqczkvcsr9TpcCj-yPKTHEDHMeCnvV9ap0Ztobk-xlJ6eD6jlhUjh4Krg0XladQhGhoKGtb7sBgwA1R0GUMc-vA-EqIg0GDYakaJLT5A"

In [24]:
def load_cached_token():
    """Read the token back out from its cache file."""
    try:
        with open(TOKEN_CACHE, "r") as cache_file:
            data = json.loads(cache_file.read())
            token = data.get("access_token")
            expires = datetime.strptime(data.get("expires"), "%Y-%m-%dT%H:%M:%S.%f%z")
            return token, expires
    except FileNotFoundError:
        return None, None

def get_token():
    """Fetch a token from the API or prompt user for credentials if needed."""
    token, expires = load_cached_token()
    now = datetime.now(timezone.utc)

    if token and expires and expires > now:
        print("Using valid cached CEDA token.")
        return token

    print("Token missing or expired. Requesting new token...")
    username = input("CEDA Username: ")
    password = getpass("CEDA Password: ")

    auth_string = f"{username}:{password}".encode("utf-8")
    b64_auth = b64encode(auth_string).decode("utf-8")

    headers = {"Authorization": f"Basic {b64_auth}"}
    response = requests.post(TOKEN_URL, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Failed to obtain token: {response.text}")

    data = response.json()
    with open(TOKEN_CACHE, "w") as cache_file:
        cache_file.write(json.dumps(data))

    return data.get("access_token")

def download_and_process(url, var_id, token):
    """Fetch the specific NetCDF file from the stream and save it locally."""
    headers = {"Authorization": f"Bearer {token}"}
    print(f"Connecting to: {url}")
    
    response = requests.get(url, headers=headers, stream=True)
    
    if response.status_code != 200:
        print(f"Error: Failed to fetch data (Status {response.status_code})")
        return

    # Extract filename from URL to save locally
    file_dir = Path(f"./data/bronze/climate_haduk/{var_id}")
    file_dir.mkdir(parents = True, exist_ok = True)
    filename = file_dir / os.path.basename(urlparse(url).path)
    
    try:
        # Open dataset directly from memory stream
        with Dataset(filename, memory=response.content) as nc:
            if var_id in nc.variables:
                print(f"-> Successfully verified variable '{var_id}'")
                # Save locally to your notebook workspace
                with open(filename, "wb") as f:
                    f.write(response.content)
                print(f"-> Saved {filename} to disk.")
            else:
                print(f"-> Warning: Variable '{var_id}' not found in {filename}")
    except Exception as e:
        print(f"-> Error processing dataset {filename}: {e}")


In [29]:
# 1. Configuration Parameters
URL_TEMPLATE = "https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/v1.3.1.ceda/1km/{var}/mon/v20250415/{var}_hadukgrid_uk_1km_mon_{year}01-{year}12.nc?download=1"

VARIABLE_ID = "hurs"
START_YEAR = 2009
END_YEAR = 2024  # Adjusted for a shorter test run

# 2. AuthenticateW
try:
    for year in range(START_YEAR, END_YEAR + 1):
        target_url = URL_TEMPLATE.format(year = year, var = VARIABLE_ID)
        download_and_process(target_url, VARIABLE_ID, MY_CEDA_TOKEN)
        
except Exception as e:
    print(f"Process failed: {e}")

Connecting to: https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/v1.3.1.ceda/1km/hurs/mon/v20250415/hurs_hadukgrid_uk_1km_mon_200901-200912.nc?download=1
-> Successfully verified variable 'hurs'
-> Saved data\bronze\climate_haduk\hurs\hurs_hadukgrid_uk_1km_mon_200901-200912.nc to disk.
Connecting to: https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/v1.3.1.ceda/1km/hurs/mon/v20250415/hurs_hadukgrid_uk_1km_mon_201001-201012.nc?download=1
-> Successfully verified variable 'hurs'
-> Saved data\bronze\climate_haduk\hurs\hurs_hadukgrid_uk_1km_mon_201001-201012.nc to disk.
Connecting to: https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/v1.3.1.ceda/1km/hurs/mon/v20250415/hurs_hadukgrid_uk_1km_mon_201101-201112.nc?download=1
-> Successfully verified variable 'hurs'
-> Saved data\bronze\climate_haduk\hurs\hurs_hadukgrid_uk_1km_mon_201101-201112.nc to disk.
Connecting to: https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insi